In [0]:
dbutils.widgets.removeAll()

In [0]:
%sql
create widget text catalogo default "catalog_au";
create widget text esquema_source default "bronze";
create widget text esquema_sink default "silver";

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

df_concepts_excel = spark.table(f"{catalogo}.{esquema_source}.conceptos_negocio")
df_rules = spark.table(f"{catalogo}.{esquema_source}.reglas_calidad")
df_trace = spark.table(f"{catalogo}.{esquema_source}.trazabilidad")
df_app_custom = spark.table(f"{catalogo}.{esquema_source}.app_custom_concepts")

In [0]:
# Filtrar registros nulos o que son de la cabecera (explicaciones de columnas en el archivo original)
df_concepts_excel = df_concepts_excel.filter(
    (col("codigo_de_entidad_del_dato").isNotNull()) & 
    (col("codigo_de_entidad_del_dato") != "Identificador único secuencial, asignado al Entidad de Dato por Data Officer") &
    (trim(col("codigo_de_entidad_del_dato")) != "")
).withColumn("codigo_de_entidad_del_dato", trim(col("codigo_de_entidad_del_dato")))

df_rules = df_rules.filter(
    (col("id_regla_de_calidad").isNotNull()) & 
    (col("id_regla_de_calidad") != "Identificador de Regla de Calidad") &
    (trim(col("id_regla_de_calidad")) != "")
).withColumn("termino_de_negocio", trim(col("termino_de_negocio")))

df_trace = df_trace.filter(
    (col("codigo_de_entidad_del_dato").isNotNull()) & 
    (col("codigo_de_entidad_del_dato") != "Identificador único secuencial, asignado al Entidad de Dato por CDO") &
    (trim(col("codigo_de_entidad_del_dato")) != "")
).withColumn("codigo_de_entidad_del_dato", trim(col("codigo_de_entidad_del_dato")))

# Normalizar el ID numérico para el join (CRM00001 → 1, CRM001 → 1, 001 → 1)
df_concepts_excel = df_concepts_excel.withColumn("_id_num", regexp_extract(col("codigo_de_entidad_del_dato"), r"(\d+)$", 1).cast("int"))
df_trace = df_trace.withColumn("_id_num", regexp_extract(col("codigo_de_entidad_del_dato"), r"(\d+)$", 1).cast("int"))

In [0]:
# Añadir bandera indicativa de origen
df_concepts_excel = df_concepts_excel.withColumn("is_custom_concept", lit(False))
df_app_custom = df_app_custom.withColumn("is_custom_concept", lit(True))

# Alinear esquemas de la tabla de la App para hacer UNION (rellenar columnas no provistas por la App con nulo)
for col_name in df_concepts_excel.columns:
    if col_name not in df_app_custom.columns:
        df_app_custom = df_app_custom.withColumn(col_name, lit(None).cast(StringType()))

# Seleccionar las mismas columnas en el mismo orden
df_app_custom_aligned = df_app_custom.select(*df_concepts_excel.columns)

# Unir ambos origenes
df_concepts_all = df_concepts_excel.union(df_app_custom_aligned)

In [0]:
# 1. Cruzar Conceptos con Trazabilidad (por codigo_de_entidad_del_dato)
df_joined_trace = df_concepts_all.alias("c").join(
    df_trace.alias("t"),
    col("c._id_num") == col("t._id_num"),
    "left"
)

# 2. Cruzar con Reglas de Calidad (por termino_de_negocio)
df_silver_pre = df_joined_trace.join(
    df_rules.alias("r"),
    col("c.termino_de_negocio") == col("r.termino_de_negocio"),
    "left"
)

In [0]:
df_silver_final = df_silver_pre.select(
    col("c.codigo_de_entidad_del_dato").alias("codigo_de_entidad_del_dato"),
    col("c.concepto_de_negocio").alias("concepto_de_negocio"),
    col("c.termino_de_negocio").alias("termino_de_negocio"),
    col("c.nombre_del_dominio").alias("nombre_del_dominio"),
    col("c.nombre_del_subdominio").alias("nombre_del_subdominio"),
    col("c.data_owner").alias("data_owner"),
    col("c.dato_critico").alias("dato_critico"),
    col("c.personal").alias("personal"),
    col("c.sensible").alias("sensible"),
    col("c.prioridad_del_entidad_de_dato").alias("prioridad_del_entidad_de_dato"),
    col("r.id_regla_de_calidad").alias("id_regla_de_calidad"),
    col("r.descripcion_de_regla_de_calidad").alias("descripcion_de_regla_de_calidad"),
    col("r.principio_de_calidad_asociado").alias("principio_de_calidad_asociado"),
    col("r.umbral_superior").cast(DoubleType()).alias("umbral_superior"),
    col("r.umbral_inferior").cast(DoubleType()).alias("umbral_inferior"),
    col("t.tabla_en_fuente_oficial").alias("tabla_en_fuente_oficial"),
    col("t.campo_en_fuente_oficial").alias("campo_en_fuente_oficial"),
    col("t.aplicativos").alias("aplicativo"),
    col("t.fuente_oficial").alias("fuente_oficial"),
    to_timestamp(col("t.fecha_de_actualizacion_en_el_diccionario_tecnico")).alias("fecha_actualizacion_diccionario"),
    col("c.is_custom_concept").alias("is_custom_concept"),
    current_timestamp().alias("ingestion_date")
)

In [0]:
target_table = f"{catalogo}.{esquema_sink}.conceptos_calidad_trazabilidad"
print(f"Escribiendo en la tabla Silver: {target_table}...")
df_silver_final.write.mode("overwrite").insertInto(target_table)